# The same ant on a Mac GPU

This notebook records the native wgpu census port. Read [README.md](README.md) for the design and [NOTEBOOK.md](NOTEBOOK.md) for the development log.

## Assumptions and predictions

One invocation owns one ant and each allocated tile. The ant is serial; only independent jobs run in parallel. Output must agree with the CPU oracle regardless of lane count, job order, slice length or memory pressure. No speedup factor was predicted for the initial implementation. Later timing measurements are recorded below.

## Provenance

Implementation revision: c0c93c981811ef96074791c0a96d439029a05ecc. Starting revision: b942a82854b4d61222301959b9895060d2543d3f. Property-test seed: 20261023. Hardware: Apple M4 Max, Metal, 40 GPU cores. The saved [validation record](results/validation.json) contains exact commands, compiler versions and SHA-256 hashes for source, binaries, inputs and outputs. This notebook has no executable cells and does not change saved results.

## Method

The shader uses sparse 64-by-64 byte tiles, four cells per 32-bit word, paired 32-bit lifetime counters and exact SplitMix64 arithmetic for backgrounds. The CPU reconstructs the final trail and runs the existing highway detector. Host tests compare every RunResult field, with an independent seeded dense reference.

## Results and controls

All seven Metal integration tests passed, including 32 property cases across three configurations. Controls include zero budgets, partial heading words, all supported colour counts, seeded high bits, tile reuse, forced pool retries, known highways and escape on the final budgeted step. A separate hardware test passed across 2^32, checking counter carry and history indexing. The full CPU suite passed 205 tests. A 57-rule census produced CSV identical to the CPU sweep. Read [validation.json](results/validation.json) for provenance, [census.csv](results/census.csv) for saved rows and [stdin.csv](results/stdin.csv) for the mixed blank/seeded CLI example. None of these results is a throughput measurement.

## Limits and next work

Only native census sweeps are ported. The CUDA probe and browser simulator are unchanged. Correctness does not establish throughput: dispatch, state readback, byte packing, paired arithmetic and host detection can limit gains. The later timing study below measures end-to-end behavior. Kernel profiling and isolated-machine repeats remain open.

## Pruning decision

The only matching record, NR-20261022-additive-laws-order-blind, concerns additive conservation laws. It does not exclude platform correctness checks. No scientific rule family is pruned and no new dead-end signature is warranted.

## Repeated wall-clock measurements

On this M4 Max, the 4083-rule, one-million-step batch was 1.84x faster on Metal than the faster CPU runner. At ten million steps, the default 256 MiB pool caused 1650–2118 retries and Metal was 2.00x slower. A 1 GiB pool removed retries and made the same workload 2.01x faster (5.068 s versus 10.192 s). The 57-rule case was 7.85x slower, and the 1013-rule, ten-million-step case was 1.59x slower. All ratios use five-run medians and the best CPU baseline per case. Desktop applications remained active. These are end-to-end measurements, not shader-only timings.

Read [BENCHMARKS.md](BENCHMARKS.md) for all ranges and [benchmark-wide-pool.json](results/benchmark-wide-pool.json) for the large-pool commands, source revision, binary/input-output hashes and samples. First-three-case harness: 655aec0; wide-case harness: f4fd86d. Local timing was authorized after CodSpeed could not install its Apple ARM harness.